In [ ]:
from pathlib import Path
import json, subprocess, sys, zipfile, shutil
INPUT=Path('/kaggle/input'); WORK=Path('/kaggle/working')
m=list(INPUT.rglob('active_tb_safe_v1/SAFE_BOUNDARY.json'))
if len(m)==1: SRC=m[0].parent
else:
 a=list(INPUT.rglob('active_tb_safe_v1.zip')); assert len(a)==1,a
 d=WORK/'att'
 with zipfile.ZipFile(a[0]) as zf: zf.extractall(d)
 SRC=d/'active_tb_safe_v1'
print(json.loads((SRC/'SAFE_BOUNDARY.json').read_text()))
subprocess.run(['apt-get','update','-qq'],check=True)
subprocess.run(['apt-get','install','-y','-qq','yosys','iverilog','z3'],check=True)
print('tools ready')


In [ ]:
for fe in ('correctbench','autobench'):
    OUT=WORK/f'safe_out_{fe}'; OUT.mkdir(exist_ok=True)
    print('='*30, fe, '='*30, flush=True)
    r=subprocess.run([sys.executable,str(SRC/'safe'/'run_safe.py'),
      '--frontend',fe,'--tasks',str(SRC/'amp'/'pilot_tasks.json'),
      '--bounds','4,8,16,32','--timeout','90','--max-pairs','6',
      '--out',str(OUT)],text=True,capture_output=True,cwd=SRC)
    print(r.stdout[-5000:])
    if r.returncode: print('STDERR:',r.stderr[-2500:])
    shutil.make_archive(str(WORK/f'safe_out_{fe}'),'zip',OUT)
print('packaged')
